In [1]:
from langgraph.graph import StateGraph

In [2]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [4]:
llm = ChatOpenAI(model="gpt-4.1-nano")

In [6]:
from typing import TypedDict, Optional

class AuthState(TypedDict):
    username: Optional[str]
    password: Optional[str]
    is_authenticated: Optional[bool]
    output: Optional[str]

Example Objects and Their States

Object 1: Successful Login

Here is an exmaple of the AuthState object with a successfullogin:


In [7]:
auth_state_1: AuthState = {
    "username": "alice123",
    "password": "123",
    "is_authenticated": True,
    "output": "Login successful."
}
print(f"auth_state_1: {auth_state_1}")

auth_state_1: {'username': 'alice123', 'password': '123', 'is_authenticated': True, 'output': 'Login successful.'}


Object 2: Unsuccessful Login

Here is an example of the AuthState object with an unsuccessful Login:

In [8]:
auth_state_2: AuthState = {
    "username":"",
    "password": "wrongpassword",
    "is_authenticated": False,
    "output": "Authentication failed. Please try again."
}
print(f"auth_state_2: {auth_state_2}")

auth_state_2: {'username': '', 'password': 'wrongpassword', 'is_authenticated': False, 'output': 'Authentication failed. Please try again.'}


In [9]:
def input_node(state):
    print(state)
    if state.get('username', "") == "":
        state['username'] = input("What is your username?")
        
    password = input("Enter your password: ")
    
    return {"password": password}

In [10]:
input_node(auth_state_1)

{'username': 'alice123', 'password': '123', 'is_authenticated': True, 'output': 'Login successful.'}


{'password': '456'}

In [11]:
input_node(auth_state_2)

{'username': '', 'password': 'wrongpassword', 'is_authenticated': False, 'output': 'Authentication failed. Please try again.'}


{'password': '123456'}

In [12]:
def validate_credentials_node(state):
    # Extract username and password from the state
    username = state.get("username", "")
    password = state.get("password", "")

    print("Username :", username, "Password :", password)
    # Simulated credential validation
    if username == "test_user" and password == "secure_password":
        is_authenticated = True
    else:
        is_authenticated = False

    # Return the updated state with authentication result
    return {"is_authenticated": is_authenticated}

In [13]:
validate_credentials_node(auth_state_1)

Username : alice123 Password : 123


{'is_authenticated': False}

In [14]:
auth_state_3: AuthState = {
    "username":"test_user",
    "password":  "secure_password",
    "is_authenticated": False,
    "output": "Authentication failed. Please try again."
}
print(f"auth_state_3: {auth_state_3}")

auth_state_3: {'username': 'test_user', 'password': 'secure_password', 'is_authenticated': False, 'output': 'Authentication failed. Please try again.'}


In [15]:
validate_credentials_node(auth_state_3)

Username : test_user Password : secure_password


{'is_authenticated': True}

In [22]:
# Define the success node
def success_node(state):
    return {"output": "Authentication successful! Welcome."}

In [23]:
success_node(auth_state_2)

{'output': 'Authentication successful! Welcome.'}

In [24]:
# Define the failure node
def failure_node(state):
    return {"output": "Not Successfull, please try again!"}

In [28]:
failure_node(auth_state_3)

{'output': 'Not Successfull, please try again!'}

Defining the Router Node

The router node acts as a decision-making point in the workflow. It takes the current state as input and determines the next node to execute based on the is_authenticated value in the state.

In [29]:
def router(state):
    if state['is_authenticated']:
        return "success_node"
    else:
        return "failure_node"

This node ensures that the graph transitions to the appropriate node -- either the success or failure node -- based on whether the authentication was successful. It is an essential part of managing conditional logic in the workflow.

# Creating the Graph

To begin building the workflow, we need to create a graph that will serve as the foundation for connecting nodes and defining the application's logic. We create a new instance of StateGraph using our AuthState structure, which acts as a blueprint for the application's state. This graph will manage the flow of execution between nodes, ensuring a seamless and organized workflow

In [30]:
from langgraph.graph import StateGraph
from langgraph.graph import END

# Create an instance of StateGraph with the GraphState structure
workflow = StateGraph(AuthState)

In [31]:
workflow

## Adding Nodes to the Graph

Now, we add nodes to the graph to define the tasks and logic of the workflow. Nodes are added using the add_node method, which takes two arguments:
  1.  Node Name: A unique string identifier for the node.'
  2.  Node Function: The function that will execute the logic for this node.

To gather user input for authentication, we add the input_node to the graph using the add_node method. This node prompts the user to enter their username and password if they are not already present in the state.

 -- "InputNode": This is the unique identifier for the input node.
 
 -- input_node: The function that collects the username and password from the user and updates the state accordingly.


In [32]:
workflow.add_node("InputNode", input_node)

To handle the authentication logic, we add the validate_credentials_node to the graph using the add_node method. This node validates the username and password provided by the user and updates the state with the authentication result.

In [33]:
workflow.add_node("ValidateCredential", validate_credentials_node)

Now, we add the success_node to the graph using the add_node method. This node will be triggered if the credentials are validated successfully, and it will return a success message.

In [34]:
workflow.add_node("Success", success_node)

Next, we add the failure_node to the graph using the add_node method. This node will be triggered if the credentials are invalid, returning a failure message.

In [35]:
workflow.add_node("Failure", failure_node)

## Edges

Edges define the connections between nodes and represnt the flow of execution within the graph. They dictate how the AI Agent transitions from one task to another based on predefined logic or conditions. In the authentication workflow, edges guide the application flow, determining the path taken based on the results of each nodes's execution.

In [36]:
workflow.add_edge("InputNode", "ValidateCredential")


add_edge(start, end): This method creates a directed edge between two nodes, defining the flow of execution from one node to another.

-- start: The node from which the flow begins. In this case, it's "InputNode", where the user provides their credentials.

-- end: The node to which the flow leads. Here, it's "ValidateCredential", where the credentials entered by the user are validated.

In [37]:
workflow.add_edge("Success", END)

In [38]:
workflow.add_edge("Failure", "InputNode")

## Conditional Edges

Conditional egdes enable decision-making by allowing transitions between nodes based on specific conditions within the state. These edges define the flow of execution based on outcomes such as user input, validation results or any other predefined logic. By using conditional edges, the AI agent can dynamically choose its path based on the results of previous tasks.


In [ ]:
workflow.add_conditional_edges("ValidateCredential", router, {"success_node": "Success", "failure_node": "Failure"})

add_conditional_edges(start, router, conditions): This method defines the conditional transitions from a given node.

-- start: The node where the conditional egdes start (in this case, "ValidateCredential").

-- router: A function that determines the condition. It checks the current state (like the is_authenticated status) and returns the appropriate node to transition to (either "Success" or "Failure").

-- conditions: A dictionary that maps conditions (such as "success_node or "failure_node") to target nodes, indicating where to direct the flow based on the condition.

In [40]:
workflow.set_entry_point("InputNode")

set_entry_point(node): This method sets the starting point of the workflow.

-- node: The name of the node where the workflow will begin. In this case, "InputNode", which ensures the agent prompts the user for their credentials before proceeding with the authentication process.

In [41]:
app = workflow.compile()

In [43]:
inputs = {"username": "test_user"}
result = app.invoke(inputs)
print(result)

{'username': 'test_user'}
Username : test_user Password : 1234
{'username': 'test_user', 'password': '1234', 'is_authenticated': False, 'output': 'Not Successfull, please try again!'}
Username : test_user Password : 1234
{'username': 'test_user', 'password': '1234', 'is_authenticated': False, 'output': 'Not Successfull, please try again!'}
Username : test_user Password : secure_password
{'username': 'test_user', 'password': 'secure_password', 'is_authenticated': True, 'output': 'Authentication successful! Welcome.'}


In [44]:
result['output']

'Authentication successful! Welcome.'